# 02 — Dataset Validation
## SmartMine Vision AI · Stage 1: PPE Detection

---

### Objectives

1. Detect **corrupted or unreadable images**.
2. Find **missing label files** (image without annotation).
3. Identify **empty annotation files**.
4. Flag **invalid bounding boxes** (outside [0,1] range or zero-area).
5. Detect **duplicate files**.
6. Generate a **validation report** saved to `docs/research/`.

## 1. Setup

In [ ]:
import sys, hashlib, json
from pathlib import Path
from datetime import datetime

import cv2
import pandas as pd

PROJECT_ROOT = Path().resolve().parents[1]
sys.path.insert(0, str(PROJECT_ROOT))

from src.ppe_detection.utils import (
    TRAIN_IMAGES, TRAIN_LABELS,
    VALID_IMAGES, VALID_LABELS,
    TEST_IMAGES,  TEST_LABELS,
    DOCS_DIR, ensure_dirs,
)
ensure_dirs()
print("Ready.")

## 2. Validation Functions

In [ ]:
def check_image(img_path: Path) -> str | None:
    """Return error string if image can't be read, else None."""
    img = cv2.imread(str(img_path))
    if img is None:
        return "unreadable"
    if img.size == 0:
        return "empty_array"
    return None


def check_label(lbl_path: Path) -> list[str]:
    """Return list of issues found in a label file."""
    issues = []
    lines = lbl_path.read_text().splitlines()
    lines = [l for l in lines if l.strip()]
    if not lines:
        issues.append("empty_annotation")
        return issues
    for i, line in enumerate(lines):
        parts = line.strip().split()
        if len(parts) != 5:
            issues.append(f"line_{i}_bad_format")
            continue
        _, cx, cy, w, h = map(float, parts)
        if not (0 <= cx <= 1 and 0 <= cy <= 1 and 0 < w <= 1 and 0 < h <= 1):
            issues.append(f"line_{i}_out_of_bounds")
        if w == 0 or h == 0:
            issues.append(f"line_{i}_zero_area")
    return issues


def file_hash(path: Path, chunk: int = 65536) -> str:
    h = hashlib.md5()
    with open(path, "rb") as f:
        while data := f.read(chunk):
            h.update(data)
    return h.hexdigest()


print("Functions defined.")

## 3. Run Validation on All Splits

> This cell may take 1–2 minutes on 2601 images.

In [ ]:
splits = [
    ("train", TRAIN_IMAGES, TRAIN_LABELS),
    ("valid", VALID_IMAGES, VALID_LABELS),
    ("test",  TEST_IMAGES,  TEST_LABELS),
]

records = []
seen_hashes: dict[str, str] = {}

for split_name, img_dir, lbl_dir in splits:
    images = sorted(img_dir.glob("*.jpg")) + sorted(img_dir.glob("*.png"))
    for img_path in images:
        lbl_path = lbl_dir / (img_path.stem + ".txt")
        record = {"split": split_name, "file": img_path.name, "issues": []}

        # Image check
        img_err = check_image(img_path)
        if img_err:
            record["issues"].append(img_err)

        # Label existence
        if not lbl_path.exists():
            record["issues"].append("missing_label")
        else:
            label_issues = check_label(lbl_path)
            record["issues"].extend(label_issues)

        # Duplicate detection
        h = file_hash(img_path)
        if h in seen_hashes:
            record["issues"].append(f"duplicate_of:{seen_hashes[h]}")
        else:
            seen_hashes[h] = img_path.name

        record["issue_count"] = len(record["issues"])
        records.append(record)

val_df = pd.DataFrame(records)
print(f"Validated {len(val_df)} files.")
print(f"Files with issues: {val_df[val_df.issue_count > 0].shape[0]}")

## 4. Issue Summary

In [ ]:
problems = val_df[val_df.issue_count > 0].copy()
print(problems[["split", "file", "issues"]].to_string(index=False))

# Flatten issue types for counting
all_issues = [issue for issues in val_df["issues"] for issue in issues]
from collections import Counter
issue_counts = Counter(all_issues)
print("\nIssue type counts:")
for issue, cnt in issue_counts.most_common():
    print(f"  {issue}: {cnt}")

## 5. Save Validation Report

In [ ]:
report = {
    "generated_at": datetime.now().isoformat(),
    "dataset": "Construction Site Safety (Roboflow)",
    "total_files_checked": len(val_df),
    "files_with_issues": int(val_df[val_df.issue_count > 0].shape[0]),
    "issue_type_counts": dict(issue_counts),
    "problem_files": problems[["split","file","issues"]].to_dict(orient="records"),
}

report_path = DOCS_DIR / "ppe_dataset_validation_report.json"
report_path.write_text(json.dumps(report, indent=2))
print(f"Report saved → {report_path}")

# Also save CSV
csv_path = DOCS_DIR / "ppe_validation_detail.csv"
val_df.to_csv(csv_path, index=False)
print(f"Detail CSV saved → {csv_path}")

## 6. Conclusions & Next Steps

**What to look for:**
- If `files_with_issues == 0` → dataset is clean, proceed to training.
- If `missing_label` > 0 → those images will be ignored by YOLO automatically.
- If `empty_annotation` > 0 → background images (useful for reducing false positives).
- If `duplicate` > 0 → consider removing to avoid data leakage between splits.

**Next:** `03_training_yolo.ipynb` — fine-tune YOLOv8n on the validated dataset.